In [1]:
# Part1
# upload the docuemnts
# make the chunking
# make the embedding
# store the embedding to the vector db

# Part2
# Initiate llm model
# Invoke embedding
# merge embedding + user question and pass to llm invoke method


In [21]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
EMBEDDING_MODEL = "text-embedding-3-large"
# EMBEDDING_MODEL = "all-MiniLM-L6-v2"
db_name = "vector_db"

In [4]:
filename = "/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar_Sharma_Resume.pdf"
loader = PyPDFLoader(filename)
documents = loader.load()


In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
len(chunks)
chunks[0].page_content


'https://www.linkedin.com/in/pravar-sharma-3410a199/ \nhttps://github.com/pravar1919\npravar.sharma@gmail.com\n+91 9414472171\nPravarSharma\nWork Experience\nSenior Engineer - Cloud Services & Software\nLTIMindtree | Sept 2023 - Present\nDeveloped new features and enhanced existing functionalities to meet evolving client\nrequirements.\nBuilt a FastAPI-based service to automate JIRA API for generating and downloading reports.\nIntegrated OAuth authentication for secure API access and data retrieval.'

In [6]:
chunks[1].page_content

'Integrated OAuth authentication for secure API access and data retrieval.\nCollaborated with cross-functional teams to deliver high-quality, scalable solutions.\nPython Developer\nQuixom Technology (Trootech Business Solutions Pvt. Ltd.) | Sept 2022 - July 2023\nLed Zoho Projects, Chats, and user data migration to a locally deployed Mattermost instance.\nEnsured zero downtime deployment and seamless data transfer with system integrity.'

In [7]:
chunks[2].page_content

'Ensured zero downtime deployment and seamless data transfer with system integrity.\nWorked as a dedicated resource on a microservices architecture, utilizing queues, Docker, and\ncloud functions.\nPython Developer\nIdeepeners Pvt. Ltd. | April 2021 - August 2022\nDeveloped a SaaS analytics platform for Shopify stores integrating Google Analytics and Google\nAds.\nBuilt Django-based RESTful APIs and dynamic UI dashboards for real-time insights.'

In [8]:
# embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 12 documents


In [22]:
# Part 2
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [25]:
# llm = ChatOpenAI(temperature=0, model="gpt-5-nano")
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [11]:
retriver = vectorstore.as_retriever()

In [12]:
retriver.invoke("what is the name?")

[Document(id='b018050d-9c55-4592-88fa-9f9126d02a87', metadata={'page': 1, 'source': '/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar_Sharma_Resume.pdf', 'page_label': '2', 'moddate': '2025-02-03T04:35:57+00:00', 'total_pages': 2, 'creationdate': '2025-02-03T04:35:57+00:00', 'creator': 'Canva', 'author': 'pravar sharma', 'keywords': 'DAFdGVhcxDo,BAEOIxIzBgU,0', 'producer': 'Canva', 'title': 'Pravar Sharma'}, page_content='Siavite | Event Management SaaS\nBuilt a web-based platform for managing events, vendors, and customer bookings.\nDeveloped a vendor earnings dashboard with real-time analytics and automated invoicing.\nTech Stack: Django, PostgreSQL, AWS RDS, Twilio, Stripe, jQuery\nPost Graduate Program in Cloud Computing (PGPCC)\nGreat Lakes Institute of Management | Dec 2022\nBachelor of Technology (B.Tech) in Electrical Engineering\nRajasthan Technical University | 2013\nEducation'),
 Document(id='41db1f32-0d44-4258-ac78-5bb19bf4ac5e', metadata={'creator': 'Canva

In [13]:
llm.invoke("what is the name?")

AIMessage(content="Could you please provide more context or specify what you're referring to?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 12, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_1a97b5aa6c', 'id': 'chatcmpl-CiFbZ1c4IgrYRGKOlZYzQzf0b7vd9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--e2d18ef7-be5c-4497-aa01-3d4f26cf87c9-0', usage_metadata={'input_tokens': 12, 'output_tokens': 13, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [14]:
# Now will combine retriver output + the user input and fed that to the llm.
system_prompt_template = """
    You are a knowledgeable, friendly assistant representing the resume of a candiate.
    You are chatting with a user about the resume of a candiate.
    If relevant, use the given context to answer any question.
    If you don't know the answer, say so.
    Context:
    {context}
"""

In [15]:
def answer_question(question, history):

    # Retrieve docs (RAG)
    docs = retriver.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Start message list with system prompt
    messages = [
        SystemMessage(content=system_prompt_template.format(context=context))
    ]

    # Convert Gradio ChatInterface history -> LLM format
    for msg in history:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))

    # Add latest question
    messages.append(HumanMessage(content=question))

    # LLM response
    response = llm.invoke(messages)

    # Return ONLY reply string (ChatInterface updates history itself)
    return response.content


In [16]:
answer_question("is the candidate is suitable for generative AI profile?", [])

'Based on the provided resume, the candidate has a strong background in backend development, cloud computing, API development, and microservices architecture. However, there is no specific mention of experience or expertise in generative AI, machine learning, or deep learning, which are typically essential for a generative AI profile.\n\nIf the role requires expertise in generative AI, such as working with neural networks, natural language processing, or related AI frameworks, the candidate may need additional training or experience in those areas. \n\nWould you like me to highlight any relevant skills or experiences that could be transferable to a generative AI role?'

In [17]:
import gradio as gr

In [26]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
